In [3]:
# Import all libraries we'll need
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib
import faiss
import pickle
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler

# Load the XGBoost model (our best predictive model)
xgb_model = joblib.load("../models/xgboost_v2_mice.pkl")

# Load the RAG components (vector index + guideline documents)
rag_index = faiss.read_index("../models/guideline_index.faiss")
with open("../models/guideline_documents.pkl", "rb") as f:
    guideline_documents = pickle.load(f)

# Load the sentence embedder (used for RAG queries)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Load test data (patients the model has never seen during training/validation)
test_df = pd.read_parquet("../data/processed/test_final.parquet")

print("All components loaded successfully")
print("Test data shape:", test_df.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

All components loaded successfully
Test data shape: (236654, 73)


In [4]:
# Find patients in the test set who actually developed sepsis
septic_patient_ids = test_df[test_df["SepsisLabel"] == 1]["patient_id"].unique()
print("Number of septic patients in test set:", len(septic_patient_ids))

# Pick the first one for our demo
demo_patient_id = septic_patient_ids[0]

# Get this patient's full data, sorted by time
demo_patient_data = test_df[test_df["patient_id"] == demo_patient_id].sort_values("ICULOS")

print("Demo patient ID:", demo_patient_id)
print("Total hours of data:", len(demo_patient_data))
print("\nLast 5 hours of data:")
print(demo_patient_data[["ICULOS", "HR", "MAP", "Resp", "SepsisLabel"]].tail())

Number of septic patients in test set: 458
Demo patient ID: 55
Total hours of data: 9

Last 5 hours of data:
      ICULOS    HR   MAP  Resp  SepsisLabel
2216       5  85.0  90.0  14.0            1
2217       6  91.0  95.0  19.0            1
2218       7  93.0  90.0  16.0            1
2219       8  94.0  88.0  15.0            1
2220       9  93.0  79.0  15.0            1


In [5]:
def sepsis_prediction_pipeline(patient_row, feature_cols):
    # Step 1: Get risk prediction from XGBoost
    X = patient_row[feature_cols].values.reshape(1, -1)
    risk_score = xgb_model.predict_proba(X)[0, 1]
    print("=== STEP 1: Risk Prediction ===")
    print(f"Sepsis Risk Score: {risk_score*100:.1f}%\n")

    # Step 2: Identify key clinical findings (simple threshold-based flagging)
    key_findings = []
    if patient_row["Lactate_measured"] == 1:
        key_findings.append("lactate testing was performed, suggesting clinical concern")
    if patient_row["HR"] > 90:
        key_findings.append("elevated heart rate")
    if patient_row["MAP"] < 70:
        key_findings.append("low mean arterial pressure")
    if patient_row["Resp"] >= 22:
        key_findings.append("elevated respiratory rate")
    if patient_row["ShockIndex"] > 0.9:
        key_findings.append("elevated shock index")
    if not key_findings:
        key_findings.append("general vital sign patterns")

    query = ", ".join(key_findings)
    print("=== STEP 2: Key Clinical Findings ===")
    print(f"{query}\n")

    # Steps 3 & 4: Retrieve relevant guidelines using RAG
    query_embedding = embedder.encode([query], normalize_embeddings=True)
    scores, indices = rag_index.search(np.array(query_embedding), 3)

    retrieved_docs = []
    for idx, score in zip(indices[0], scores[0]):
        retrieved_docs.append({"document": guideline_documents[idx], "similarity_score": float(score)})

    print("=== STEP 3: Retrieved Clinical Evidence ===")
    for i, doc in enumerate(retrieved_docs):
        print(f"[{i+1}] (similarity: {doc['similarity_score']:.3f})")
        print(f"{doc['document']}\n")

    # Step 5: Generate final explanation
    explanation = "=== STEP 4: Final Explanation ===\n"
    explanation += f"This patient has a {risk_score*100:.1f}% predicted risk of sepsis, "
    explanation += f"driven by {query} [1-{len(retrieved_docs)}]. "
    explanation += "Clinical correlation and physician judgment are recommended before acting on this assessment."
    print(explanation)

    return {
        "risk_score": risk_score,
        "key_findings": query,
        "retrieved_evidence": retrieved_docs,
        "explanation": explanation
    }

print("Pipeline function ready")

Pipeline function ready


In [6]:
# Feature columns (same as used in training)
exclude_cols = ["patient_id", "SepsisLabel"]
feature_cols = [c for c in test_df.columns if c not in exclude_cols]

# Last hour of data for our demo patient
patient_last_hour = demo_patient_data.iloc[-1]

print(f"Running full pipeline for Patient ID: {demo_patient_id}, Hour: {patient_last_hour['ICULOS']}")
print("=" * 60)

result = sepsis_prediction_pipeline(patient_last_hour, feature_cols)

Running full pipeline for Patient ID: 55, Hour: 9.0
=== STEP 1: Risk Prediction ===
Sepsis Risk Score: 36.5%

=== STEP 2: Key Clinical Findings ===
elevated heart rate

=== STEP 3: Retrieved Clinical Evidence ===
[1] (similarity: 0.510)
Tachycardia, defined as a heart rate greater than 90 beats per minute, is a common early compensatory response to infection as the body attempts to maintain cardiac output despite falling blood pressure. Persistent tachycardia despite fluid resuscitation may indicate ongoing hypovolemia or shock.

[2] (similarity: 0.480)
Tachypnea, or an elevated respiratory rate above 22 breaths per minute, often reflects the body's attempt to compensate for metabolic acidosis caused by tissue hypoxia in early sepsis. It is one of the earliest and most sensitive signs of clinical deterioration.

[3] (similarity: 0.414)
The Shock Index, calculated as heart rate divided by systolic blood pressure, is a simple bedside tool for early identification of hemodynamic compromis

In [7]:
import json

demo_result_summary = {
    "patient_id": int(demo_patient_id),
    "hour": float(patient_last_hour["ICULOS"]),
    "risk_score": float(result["risk_score"]),
    "key_findings": result["key_findings"],
    "num_evidence_retrieved": len(result["retrieved_evidence"]),
    "final_explanation": result["explanation"]
}

with open("../reports/demo_pipeline_example.json", "w") as f:
    json.dump(demo_result_summary, f, indent=2)

print("Demo pipeline result saved!")

Demo pipeline result saved!


In [8]:
def sepsis_agent(patient_row, feature_cols, uncertainty_model=None, low_risk_threshold=0.3, uncertainty_threshold=0.02):
    # DECISION POINT 1: Get risk score
    X = patient_row[feature_cols].values.reshape(1, -1)
    risk_score = xgb_model.predict_proba(X)[0, 1]
    print(f"[AGENT] Step 1: Risk score = {risk_score*100:.1f}%")

    # DECISION POINT 2: If risk is low, the agent stops here (no RAG call)
    if risk_score < low_risk_threshold:
        print(f"[AGENT] Decision: Risk is below threshold ({low_risk_threshold*100}%). Skipping further analysis.")
        return {
            "risk_score": risk_score,
            "action": "MONITOR",
            "explanation": f"Risk score {risk_score*100:.1f}% is below the concern threshold. Continue routine monitoring. No further evidence retrieval needed."
        }

    print("[AGENT] Decision: Risk exceeds threshold. Proceeding to evidence retrieval.")

    # Extract key clinical findings
    key_findings = []
    if patient_row["HR"] > 90:
        key_findings.append("elevated heart rate")
    if patient_row["MAP"] < 70:
        key_findings.append("low mean arterial pressure")
    if patient_row["Resp"] >= 22:
        key_findings.append("elevated respiratory rate")
    if patient_row["ShockIndex"] > 0.9:
        key_findings.append("elevated shock index")
    if not key_findings:
        key_findings.append("general vital sign patterns")
    query = ", ".join(key_findings)

    # DECISION POINT 3: Is the prediction borderline? (distance from 0.5 as a simple proxy)
    confidence_margin = abs(risk_score - 0.5)
    is_uncertain = confidence_margin < uncertainty_threshold

    if is_uncertain:
        print(f"[AGENT] Decision: Prediction is borderline/uncertain (margin={confidence_margin:.3f}). Retrieving MORE evidence (top-5) for extra context.")
        top_k = 5
    else:
        print(f"[AGENT] Decision: Prediction is confident (margin={confidence_margin:.3f}). Retrieving standard evidence (top-3).")
        top_k = 3

    # Retrieve evidence (amount depends on the agent's decision above)
    query_embedding = embedder.encode([query], normalize_embeddings=True)
    scores, indices = rag_index.search(np.array(query_embedding), top_k)

    retrieved_docs = []
    for idx, score in zip(indices[0], scores[0]):
        retrieved_docs.append({"document": guideline_documents[idx], "similarity_score": float(score)})

    # Generate final explanation
    explanation = f"Risk: {risk_score*100:.1f}%. Findings: {query}. "
    explanation += f"Supported by {len(retrieved_docs)} pieces of clinical evidence [1-{len(retrieved_docs)}]. "
    if is_uncertain:
        explanation += "NOTE: This prediction is borderline - extra evidence provided due to model uncertainty. Close clinical correlation strongly recommended."
    else:
        explanation += "Clinical correlation and physician judgment are recommended."

    return {
        "risk_score": risk_score,
        "action": "ALERT_WITH_EXTRA_EVIDENCE" if is_uncertain else "ALERT",
        "key_findings": query,
        "retrieved_evidence": retrieved_docs,
        "explanation": explanation
    }

print("Agentic controller ready")

Agentic controller ready


In [9]:
print("=" * 60)
print("TEST 1: Demo Patient (Patient 55)")
print("=" * 60)

result_agent = sepsis_agent(patient_last_hour, feature_cols)
print("\nFinal Action:", result_agent["action"])
print("Final Explanation:", result_agent["explanation"])

TEST 1: Demo Patient (Patient 55)
[AGENT] Step 1: Risk score = 36.5%
[AGENT] Decision: Risk exceeds threshold. Proceeding to evidence retrieval.
[AGENT] Decision: Prediction is confident (margin=0.135). Retrieving standard evidence (top-3).

Final Action: ALERT
Final Explanation: Risk: 36.5%. Findings: elevated heart rate. Supported by 3 pieces of clinical evidence [1-3]. Clinical correlation and physician judgment are recommended.


In [10]:
print("=" * 60)
print("TEST 2: Low-Risk Patient")
print("=" * 60)

non_septic_patients = test_df[test_df["patient_id"].isin(
    test_df[test_df["SepsisLabel"] == 0]["patient_id"].unique()
)]

low_risk_patient_data = non_septic_patients[
    non_septic_patients["patient_id"] == non_septic_patients["patient_id"].iloc[0]
].sort_values("ICULOS")
low_risk_row = low_risk_patient_data.iloc[0]  # first hour, likely stable

result_agent_2 = sepsis_agent(low_risk_row, feature_cols)
print("\nFinal Action:", result_agent_2["action"])
print("Final Explanation:", result_agent_2["explanation"])

TEST 2: Low-Risk Patient
[AGENT] Step 1: Risk score = 17.2%
[AGENT] Decision: Risk is below threshold (30.0%). Skipping further analysis.

Final Action: MONITOR
Final Explanation: Risk score 17.2% is below the concern threshold. Continue routine monitoring. No further evidence retrieval needed.


In [11]:
print("=" * 60)
print("TEST 3: Searching for a Borderline/Uncertain Case")
print("=" * 60)

sample_rows = test_df.sample(500, random_state=1)
X_sample = sample_rows[feature_cols].values
risk_scores_sample = xgb_model.predict_proba(X_sample)[:, 1]

sample_rows = sample_rows.copy()
sample_rows["risk"] = risk_scores_sample
sample_rows["margin"] = abs(sample_rows["risk"] - 0.5)

borderline_row = sample_rows.sort_values("margin").iloc[0]
print(f"Found borderline case with risk = {borderline_row['risk']*100:.1f}%\n")

result_agent_3 = sepsis_agent(borderline_row, feature_cols)
print("\nFinal Action:", result_agent_3["action"])
print("Final Explanation:", result_agent_3["explanation"])

TEST 3: Searching for a Borderline/Uncertain Case
Found borderline case with risk = 50.0%

[AGENT] Step 1: Risk score = 50.0%
[AGENT] Decision: Risk exceeds threshold. Proceeding to evidence retrieval.
[AGENT] Decision: Prediction is borderline/uncertain (margin=0.000). Retrieving MORE evidence (top-5) for extra context.

Final Action: ALERT_WITH_EXTRA_EVIDENCE
Final Explanation: Risk: 50.0%. Findings: elevated heart rate. Supported by 5 pieces of clinical evidence [1-5]. NOTE: This prediction is borderline - extra evidence provided due to model uncertainty. Close clinical correlation strongly recommended.


In [12]:
agentic_demo_results = {
    "test_1_confident_alert": {
        "risk": float(result_agent["risk_score"]),
        "action": result_agent["action"],
        "evidence_count": len(result_agent.get("retrieved_evidence", []))
    },
    "test_2_low_risk_monitor": {
        "risk": float(result_agent_2["risk_score"]),
        "action": result_agent_2["action"],
        "evidence_count": 0
    },
    "test_3_uncertain_extra_evidence": {
        "risk": float(result_agent_3["risk_score"]),
        "action": result_agent_3["action"],
        "evidence_count": len(result_agent_3["retrieved_evidence"])
    }
}

with open("../reports/agentic_ai_demo.json", "w") as f:
    json.dump(agentic_demo_results, f, indent=2)

print("Agentic AI demo results saved!")

Agentic AI demo results saved!


In [13]:
# Every run picks a new random patient from the test set
random_patient = test_df.sample(1).iloc[0]
live_result = sepsis_agent(random_patient, feature_cols)
print("\nFinal Action:", live_result["action"])
print("Final Explanation:", live_result["explanation"])

[AGENT] Step 1: Risk score = 54.7%
[AGENT] Decision: Risk exceeds threshold. Proceeding to evidence retrieval.
[AGENT] Decision: Prediction is confident (margin=0.047). Retrieving standard evidence (top-3).

Final Action: ALERT
Final Explanation: Risk: 54.7%. Findings: low mean arterial pressure. Supported by 3 pieces of clinical evidence [1-3]. Clinical correlation and physician judgment are recommended.
